In [1]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from sklearn.metrics import roc_auc_score

In [2]:
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit')

In [103]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])

In [104]:
base.head()

,SK_ID_CURR,TARGET
0,100002,1
1,100003,0
2,100004,0
3,100006,0
4,100007,0


In [105]:
base = base.astype({'SK_ID_CURR': 'string'})

In [47]:
bureau_processed = pd.read_parquet('src\\intermediate\\bureau_processed.parquet')

In [48]:
bureau_processed_with_target = base.merge(bureau_processed, on='SK_ID_CURR', how='left')

In [9]:
def ttest_one_var(test_data, feature_col, target_col, equal_var=False):
    """
    Perform an independent t-test for one numeric column against a binary target.
    
    Args:
        test_data (pd.dataframe): Numeric data to test
        feature_col (column name of feature): Numeric feature to test
        target_col (column name of target): Binary target (must have exactly 2 groups)  
        equal_var (bool): If False (default), uses Welch's t-test (unequal variances)
    
    Returns:
        dict: t-statistic, p-value, group means, and effect size (Cohen's d)
    
    Example:
        results = ttest_one_var(df, 'LTV', 'TARGET', equal_var=False)
    """
    # Check for exactly 2 groups
    groups = [0, 1]
    
    # Split data into two groups
    group1 = test_data.loc[test_data[target_col] == groups[0], feature_col].dropna()
    group2 = test_data.loc[test_data[target_col] == groups[1], feature_col].dropna()
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=equal_var)
    
    # Calculate effect size (Cohen's d)
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1+n2-2))
    cohens_d = (group1.mean() - group2.mean()) / pooled_std
    
    return {
        'feature': feature_col,
        't_statistic': t_stat,
        'p_value': p_value,
        'mean_group1': group1.mean(),
        'mean_group2': group2.mean(),
        'cohens_d': cohens_d,
        'abs_t_stat': abs(t_stat)  # Useful for ranking variables
    }

In [10]:
ttest_one_var(bureau_processed_with_target, 'BUREAU_TOTAL_CREDIT_COUNT', 'TARGET', equal_var=False)

{'feature': 'BUREAU_TOTAL_CREDIT_COUNT',
 't_statistic': np.float64(-1.9694964045489756),
 'p_value': np.float64(0.048907863353838),
 'mean_group1': np.float64(5.556055988121239),
 'mean_group2': np.float64(5.62254516889238),
 'cohens_d': np.float64(-0.015187567390226085),
 'abs_t_stat': np.float64(1.9694964045489756)}

In [15]:
def calculate_auc_one_var(df, numeric_col, target_col):
    """
    Simplified AUC calculation using sklearn's roc_auc_score.
    Returns json with AUC scores.
    """
    x = df[numeric_col]
    y = df[target_col]
    valid_idx = x.notna() & y.notna()  # Drop missing values

    try:
        auc = roc_auc_score(y[valid_idx], x[valid_idx])
        gini_abs = abs(2 * auc - 1)  # Convert AUC to Gini coefficient
        result = {'variable': numeric_col, 'auc_score': auc, 'gini_abs': gini_abs}
    except ValueError:
        result = {'variable': numeric_col, 'auc_score': None, 'gini_abs': None}
    
    return result

In [16]:
calculate_auc_one_var(bureau_processed_with_target, 'BUREAU_TOTAL_CREDIT_COUNT', 'TARGET')

{'variable': 'BUREAU_TOTAL_CREDIT_COUNT',
 'auc_score': 0.4951090161614231,
 'gini_abs': 0.00978196767715378}

In [20]:
def calculate_numeric_selection_metrics(df, numeric_cols, target_col):
    """
    Calculate AUC and t-test for a numeric feature against a binary target.
    
    Args:
        df (pd.DataFrame): DataFrame containing the feature and target
        numeric_col (str): Name of the numeric feature column
        target_col (str): Name of the binary target column
    
    Returns:
        dict: AUC score and t-test results
    """

    results = []
    for col in numeric_cols:
        auc_result = calculate_auc_one_var(df, col, target_col)
        ttest_result = ttest_one_var(df, col, target_col, equal_var=False)
        missing_perc = df[col].isna().sum()/df.shape[0] * 100
        results.append({
            'variable': col,
            'missing_count': missing_perc,
            'auc_score': auc_result['auc_score'],
            'gini_abs': auc_result['gini_abs'],
            't_statistic': ttest_result['t_statistic'],
            'p_value': ttest_result['p_value'],
            'mean_group1': ttest_result['mean_group1'],
            'mean_group2': ttest_result['mean_group2'],
            'cohens_d': ttest_result['cohens_d']
        })
    
    return pd.DataFrame(results)
    

In [49]:
numcols = bureau_processed_with_target.select_dtypes(include=['float64']).columns.to_list()

In [21]:
numeric_selection_metrics = calculate_numeric_selection_metrics(bureau_processed_with_target, numcols, 'TARGET')

C:\Users\domjo\AppData\Local\Temp\ipykernel_10640\46911737.py:30: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std


In [26]:
columns_with_nulls = ['BUREAU_AVG_ANNUITY_PERCENT', 'BUREAU_MAX_CREDIT_OVERDUE_AMOUNT', ]

In [107]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [108]:
def logistic_regression_coef(df, feature_cols, target_col):
    """
    Fit a logistic regression model and return coefficients for each feature.
    
    Args:
        df (pd.DataFrame): DataFrame containing features and target
        feature_cols (list): List of feature column names
        target_col (str): Name of the target column
    
    Returns:
        pd.DataFrame: DataFrame with feature names and their coefficients
    """
    X = df[feature_cols]
    y = df[target_col]

    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # Fit logistic regression model
    model = LogisticRegression(random_state=123, max_iter=10000, penalty='l1', solver='saga', C=0.1)
    model.fit(X_scaled, y)

    # Create DataFrame with coefficients
    coef_df = pd.DataFrame({'variable': feature_cols, 'coef': model.coef_[0]})
    
    return coef_df

In [110]:
coef_df = logistic_regression_coef(bureau_processed_with_target, numcols, 'TARGET')

In [ ]:
numeric_selection_metrics_full = numeric_selection_metrics.merge(coef_df,
                                                            how='left', 
                                                            right_on='variable',
                                                            left_on='variable'
                                                            )